# ♻️ Sistem Klasifikasi Jenis Sampah Berdasarkan Citra Digital
### Smart Waste Classifier (Visual Komputer Cerdas & Pengolahan Citra)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akbarlubis247/Projek-Viskom-n-citra/blob/main/coba_di_colab.ipynb)

Notebook Google Colab ini dirancang untuk menguji seluruh pipeline sistem klasifikasi sampah secara interaktif dan komprehensif:
1. **Preprocessing & Segmentasi**: Gaussian Blur, Adaptive Otsu Thresholding, Koreksi Orientasi Background, Morfologi Kontur Terbesar.
2. **Ekstraksi Fitur Multi-Domain (49 Dimensi)**:
   - 🎨 **Warna (38 Dimensi)**: Histogram HSV (32 bin: 16 Hue, 8 Saturation, 8 Value) terfokus objek + Nilai Rata-rata & Standar Deviasi CIE-LAB (6 dimensi).
   - 📐 **Bentuk / Morfologi (5 Dimensi)**: Luas Relatif, Keliling, Kebundaran (Circularity), Aspect Ratio, dan Solidity.
   - 🧱 **Tekstur & Tepi (6 Dimensi)**: GLCM (Contrast, Dissimilarity, Homogeneity, Energy, Correlation) pada bounding box objek + Kerapatan Tepi Canny.
3. **Klasifikasi KNN**: Algoritma K-Nearest Neighbor dengan metrik *Manhattan Distance* & *Distance Weighting*.

---
### 📋 Menu Pengujian di Notebook Ini:
- **Langkah 1**: Clone Repository & Persiapan Dependensi
- **Langkah 2**: Quick Test 5 Sampel Bawaan (Langsung jalan tanpa upload)
- **Langkah 3**: Uji Coba Upload Foto Sampah Sendiri (Laptop/HP)
- **Langkah 4**: Uji Coba Langsung dengan Kamera Webcam Google Colab
- **Langkah 5 (Opsional)**: Evaluasi Performa Model & Confusion Matrix
- **Langkah 6 (Opsional)**: Jalankan Web App Streamlit Interaktif (via Cloudflare / Localtunnel)

## 🛠️ Langkah 1: Clone Repository & Persiapan Dependensi
Jalankan cell di bawah ini untuk mengunduh kode program, model yang sudah dilatih (`knn_model.pkl`), dan menginstal pustaka yang diperlukan.

In [ ]:
import os
import sys

REPO_NAME = "Projek-Viskom-n-citra"
REPO_URL = "https://github.com/akbarlubis247/Projek-Viskom-n-citra.git"

# 1. Pastikan direktori kerja berada pada repository
if not os.path.exists(REPO_NAME) and not os.path.exists("models"): 
    print(f"📥 Mengunduh repository dari GitHub: {REPO_URL}...")
    !git clone {REPO_URL}
    %cd {REPO_NAME}
elif os.path.exists(REPO_NAME):
    print(f"🔄 Masuk ke folder {REPO_NAME} dan memperbarui data...")
    %cd {REPO_NAME}
    !git pull
else:
    print("✅ Sudah berada di dalam direktori proyek.")

# 2. Install dependensi yang dibutuhkan
print("\n📦 Memeriksa dan menginstal paket pustaka...")
!pip install -q opencv-python scikit-learn scikit-image joblib streamlit matplotlib pyngrok

# 3. Verifikasi ketersediaan model
if os.path.exists("models/knn_model.pkl") and os.path.exists("models/scaler.pkl"):
    print("\n✅ SUKSES: Model KNN (49 fitur) dan Scaler siap digunakan!")
else:
    print("\n⚠️ Model belum ditemukan, memulai pelatihan otomatis...")
    !python train.py
    print("✅ Model berhasil dilatih dan disimpan.")

## ⚡ Langkah 2: Quick Test Menggunakan Sampah Contoh Bawaan
Jalankan cell di bawah ini untuk menguji 5 sampel sampah yang sudah tersedia di repositori:
- Botol Kaca (`sampah botol kaca.jpeg`)
- Botol Plastik (`sampah botol plastik.jpeg`)
- Kaleng (`sampah kaleng.jpg`)
- Kardus (`sampah kardus.jpeg`)
- Kertas (`sampah kertas.jpg`)

Sistem akan menampilkan **Tahap Segmentasi Citra**, **Fitur Ekstraksi**, serta **Hasil Prediksi & Grafik Confidence Bar**.

In [ ]:
import cv2
import joblib
import numpy as np
import matplotlib.pyplot as plt
from feature_extraction import extract_all_features

# Load Model dan Scaler
model = joblib.load("models/knn_model.pkl")
scaler = joblib.load("models/scaler.pkl")

LABEL_MAP = {
    "botol_plastik": "Botol Plastik 🧴",
    "botol_kaca": "Botol Kaca 🍾",
    "kaleng": "Kaleng / Logam 🥫",
    "kertas": "Kertas 📄",
    "kardus": "Kardus 📦"
}

def analyze_and_display(img_bgr, title_suffix=""):
    # 1. Ekstraksi Fitur Multi-Domain
    features, img_resized, mask = extract_all_features(img_bgr)
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    masked_obj = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)
    
    # 2. Prediksi KNN
    features_scaled = scaler.transform([features])
    prediction = model.predict(features_scaled)[0]
    nama_prediksi = LABEL_MAP.get(prediction, prediction.upper())
    
    # 3. Visualisasi 3 Panel Segmentasi
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    axes[0].imshow(img_rgb)
    axes[0].set_title("1. Citra Input (256x256)", fontsize=11, fontweight="bold")
    axes[0].axis("off")
    
    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("2. Mask Kontur Otsu & Morfologi", fontsize=11, fontweight="bold")
    axes[1].axis("off")
    
    axes[2].imshow(masked_obj)
    axes[2].set_title("3. Objek Tersegmentasi", fontsize=11, fontweight="bold")
    axes[2].axis("off")
    plt.tight_layout()
    plt.show()
    
    print(f"🎯 HASIL KLASIFIKASI: {nama_prediksi} {title_suffix}")
    print(f"   - Total Fitur: {len(features)} Dimensi (Warna HSV/LAB, Bentuk, Tekstur GLCM, Canny)")
    
    # 4. Tampilkan Grafik Probabilitas Bar Chart
    if hasattr(model, "predict_proba"):
        probs = model.predict_proba(features_scaled)[0]
        fig, ax = plt.subplots(figsize=(8, 2.2))
        classes_display = [LABEL_MAP.get(c, c) for c in model.classes_]
        y_pos = np.arange(len(classes_display))
        colors = ['#2ca02c' if c == prediction else '#1f77b4' for c in model.classes_]
        bars = ax.barh(y_pos, probs * 100, color=colors, alpha=0.85)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(classes_display, fontsize=10)
        ax.set_xlabel("Confidence Score (%)", fontsize=10)
        ax.set_xlim(0, 105)
        ax.grid(axis='x', linestyle='--', alpha=0.5)
        
        for bar in bars:
            width = bar.get_width()
            ax.text(width + 1.2, bar.get_y() + bar.get_height()/2, f"{width:.1f}%",
                    va='center', ha='left', fontsize=9, fontweight='bold')
        plt.tight_layout()
        plt.show()
    print("=" * 70)

# Jalankan evaluasi 5 sampel gambar bawaan
sample_files = [
    "sampah botol kaca.jpeg",
    "sampah botol plastik.jpeg",
    "sampah kaleng.jpg",
    "sampah kardus.jpeg",
    "sampah kertas.jpg"
]

print("🔍 Menjalankan Quick Test pada sampel gambar bawaan:\n")
for file_path in sample_files:
    if os.path.exists(file_path):
        img = cv2.imread(file_path)
        if img is not None:
            analyze_and_display(img, f"[{file_path}]")
        else:
            print(f"Gagal memuat {file_path}")

## 📤 Langkah 3: Uji Coba Upload Foto Sampah Sendiri
Jalankan cell di bawah untuk mengunggah foto sampah dari perangkat Anda (Laptop atau HP).

In [ ]:
from google.colab import files

print("Silakan klik tombol 'Choose Files' di bawah untuk mengupload foto sampah:")
uploaded = files.upload()

if not uploaded:
    print("ℹ️ Tidak ada file yang diunggah.")
else:
    for filename in uploaded.keys():
        print(f"\n📁 Memproses citra unggahan: {filename}...")
        img = cv2.imread(filename)
        if img is None:
            print(f"❌ Gagal membaca gambar: {filename}")
            continue
        analyze_and_display(img, f"[{filename}]")

## 📸 Langkah 4: Uji Coba Langsung dengan Kamera Webcam Google Colab
Cell ini memanfaatkan antarmuka webcam browser HTML5 JavaScript langsung di Google Colab:
1. Jalankan cell ini.
2. Izinkan akses kamera browser jika diminta (*Allow Camera*).
3. Arahkan benda sampah ke depan kamera, lalu klik tombol **'📸 Ambil Foto & Klasifikasi'**.

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='webcam_capture.jpg', quality=0.85):
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      div.style.marginBottom = '15px';
      
      const video = document.createElement('video');
      video.style.display = 'block';
      video.style.marginBottom = '10px';
      video.style.borderRadius = '8px';
      video.style.boxShadow = '0 3px 10px rgba(0,0,0,0.3)';
      
      const capture = document.createElement('button');
      capture.textContent = '📸 Ambil Foto & Klasifikasi';
      capture.style.background = '#28a745';
      capture.style.color = 'white';
      capture.style.padding = '10px 20px';
      capture.style.border = 'none';
      capture.style.borderRadius = '6px';
      capture.style.fontSize = '14px';
      capture.style.fontWeight = 'bold';
      capture.style.cursor = 'pointer';

      const stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});

      document.body.appendChild(div);
      div.appendChild(video);
      div.appendChild(capture);

      video.srcObject = stream;
      await video.play();

      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

try:
    print("📷 Mengaktifkan webcam...")
    captured_file = take_photo('webcam_sample.jpg')
    print("✅ Foto berhasil diambil dari webcam!\n")
    
    img = cv2.imread(captured_file)
    if img is not None:
        analyze_and_display(img, "[Hasil Webcam Langsung]")
    else:
        print("❌ Gagal memuat foto webcam.")
except Exception as e:
    print(f"Catatan: Webcam tidak dapat diakses atau dibatalkan ({e})")

## 📊 Langkah 5: (Opsional) Evaluasi Performa Model & Confusion Matrix
Jalankan cell ini untuk mengevaluasi akurasi, Precision, Recall, F1-Score, dan visualisasi Confusion Matrix Heatmap pada dataset sampah.

In [ ]:
import glob
from sklearn.metrics import classification_report, confusion_matrix

print("⏳ Mengumpulkan subset data pengujian dari folder dataset/...")
X_eval = []
y_eval = []
SAMPLES_PER_CLASS = 50

for label in model.classes_:
    folder = os.path.join("dataset", label)
    if not os.path.exists(folder):
        continue
    paths = []
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        paths.extend(glob.glob(os.path.join(folder, ext)))
    paths = paths[:SAMPLES_PER_CLASS]
    for p in paths:
        try:
            im = cv2.imread(p)
            if im is not None:
                f, _, _ = extract_all_features(im)
                X_eval.append(f)
                y_eval.append(label)
        except:
            pass

X_eval = np.array(X_eval, dtype=np.float32)
y_eval = np.array(y_eval)

print(f"✅ Berhasil memproses {len(X_eval)} sampel evaluasi dari {len(model.classes_)} kelas.\n")

# Prediksi pada data evaluasi
X_eval_scaled = scaler.transform(X_eval)
y_pred = model.predict(X_eval_scaled)

# Cetak Classification Report
target_names = [LABEL_MAP.get(c, c) for c in model.classes_]
print("=" * 60)
print("          LAPORAN EVALUASI MODEL KNN")
print("=" * 60)
print(classification_report(y_eval, y_pred, target_names=target_names))

# Visualisasi Confusion Matrix Heatmap
cm = confusion_matrix(y_eval, y_pred, labels=model.classes_)
fig, ax = plt.subplots(figsize=(6, 5))
im_plot = ax.imshow(cm, cmap="Blues")
ax.figure.colorbar(im_plot, ax=ax)
ax.set_xticks(np.arange(len(model.classes_)))
ax.set_yticks(np.arange(len(model.classes_)))
ax.set_xticklabels(target_names, rotation=35, ha="right", fontsize=10)
ax.set_yticklabels(target_names, fontsize=10)
ax.set_xlabel("Prediksi Model", fontweight="bold")
ax.set_ylabel("Kelas Aktual (Ground Truth)", fontweight="bold")
ax.set_title("Confusion Matrix Klasifikasi Sampah", fontsize=12, fontweight="bold")

for i in range(len(model.classes_)):
    for j in range(len(model.classes_)):
        color_text = "white" if cm[i, j] > cm.max() / 2 else "black"
        ax.text(j, i, cm[i, j], ha="center", va="center", color=color_text, fontweight="bold")

plt.tight_layout()
plt.show()

## 🌐 Langkah 6: (Opsional) Jalankan Web App Streamlit dari Google Colab
Anda dapat menjalankan dashboard web Streamlit interaktif langsung dari Google Colab menggunakan **Cloudflare Tunnel** (lebih stabil & tanpa password IP) atau **LocalTunnel**.

In [ ]:
# METODE UTAMA: Cloudflare Tunnel (Tanpa Password IP & Sangat Stabil)
import time

# Unduh binary cloudflared jika belum terpasang
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Jalankan Streamlit di latar belakang
!nohup streamlit run app.py --server.port 8501 --server.headless true > streamlit.log 2>&1 &
time.sleep(3)

print("🚀 Streamlit Web App berjalan!")
print("Silakan klik URL di bawah (berakhiran .trycloudflare.com) untuk membuka antarmuka web:\n")
!cloudflared tunnel --url http://127.0.0.1:8501

### Alternatif: Menjalankan dengan LocalTunnel
*(Gunakan opsi ini jika Cloudflare Tunnel dibatasi oleh jaringan Anda)*

In [ ]:
# Dapatkan IP publik Colab untuk password localtunnel
!curl -s ipv4.icanhazip.com
print("Salin nomor IP di atas untuk dimasukkan ke form LocalTunnel jika diminta.\n")

!streamlit run app.py & npx localtunnel --port 8501